# Phrase Typing Workbench

This notebook demonstrates a human-in-the-loop workflow for categorising formulaic phrases into user-defined *types* (e.g. "opening formula", "decision formula", "resumption formula"), where:

* a type can be realised by several differently-worded phrases (a "community" of variants),
* a phrase can belong to more than one type (multi-label throughout &mdash; each type is scored independently against each candidate phrase, never forcing a single partition),
* the user can browse example text windows (concordances) for any phrase before deciding,
* once a type has a few confirmed members, the tool suggests further candidates ranked by how strongly they co-occur with those members.

We use the real resolution-paragraphs corpus, picking up the same candidate phrases explored in `Phrase-Recurrence-Scale-Grouping.ipynb` &mdash; but this time letting the workbench help organise them into explicit, named types rather than just statistically-grouped scales.

In [10]:
%reload_ext autoreload
%autoreload 2

import csv
import gzip
import os
import re

from formula_detection.phrase_cooccurrence import PhraseCooccurrenceIndex
from formula_detection.phrase_typing import PhraseTypeRegistry, TypeSuggester, concordance
from formula_detection.phrase_typing_ui import PhraseTypingApp

TOKEN_RE = re.compile(r"[A-Za-z\u00c0-\u00ff]+")


def tokenize(text):
    return TOKEN_RE.findall(text.lower())


rows = []
if os.getcwd().endswith('formula_detection/docs'):
    data_dir = '../data'
else:
    data_dir = 'data'

with gzip.open(f'{data_dir}/resolution_paragraphs/paragraphs-3823.tsv.gz', 'rt', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        rows.append(row)

flat_stream = []
for row in rows:
    flat_stream.extend(tokenize(row['text']))

print(f'{len(rows)} paragraphs, {len(flat_stream)} tokens')

2909 paragraphs, 449905 tokens


## 1. Candidate phrases

A handful of phrases spanning the levels of structure described for this corpus: day-level resumption (the six Latin weekday openers plus their associated formulas), resolution-level decision formulas (including the "no action taken" variant), and two opening-formula candidates for different document types (missive vs. requeste) that haven't been assigned to any type yet.

In [11]:
day_names = ['luna den', 'martis den', 'mercurii den', 'jovis den', 'veneris den', 'sabbathi den']
candidate_phrases = day_names + [
    'genoomen zyn geleesen en geresumeert',
    'gelijk ook geresumeert en gearresteert',
    'waar op gedelibereert zynde is goedgevonden',
    'is goedgevonden en verstaan dat',
    'waar op geen resolutie is gevallen',
    'ontfangen een missive van',
    'is ter vergaderinge geleesen de requeste van',
]

cooc_index = PhraseCooccurrenceIndex(candidate_phrases, windows=[50])
cooc_index.index_stream(flat_stream)
phrase_freq = {p: len(cooc_index.phrase_positions.get(p, [])) for p in candidate_phrases}

for phrase, freq in sorted(phrase_freq.items(), key=lambda x: -x[1]):
    print(f'  {freq: >5}  {phrase}')

   1267  ontfangen een missive van
    836  waar op gedelibereert zynde is goedgevonden
    835  waar op geen resolutie is gevallen
    730  is goedgevonden en verstaan dat
    331  is ter vergaderinge geleesen de requeste van
    235  gelijk ook geresumeert en gearresteert
    223  genoomen zyn geleesen en geresumeert
     54  jovis den
     54  veneris den
     53  martis den
     50  mercurii den
     48  luna den
      1  sabbathi den


## 2. The interactive workbench

`PhraseTypingApp` has three tabs:

1. **Browse & assign** &mdash; pick any candidate phrase, read example occurrences in context, and assign it to one or more types (existing or new).
2. **Suggestions** &mdash; pick a type that already has confirmed members and a window size, and get a ranked list of still-uncategorised phrases with quick-assign buttons and mini-concordance previews.
3. **Registry overview** &mdash; a live summary of all types and their members.

Run the cell below and use the widget directly: browse a phrase, create a type (e.g. type `decision formula` in the "New type" box while `waar op gedelibereert zynde is goedgevonden` is selected, then click Assign), assign its known variants the same way, then switch to the Suggestions tab to see what else the tool proposes.

In [12]:
registry = PhraseTypeRegistry()
suggester = TypeSuggester(cooc_index.phrase_positions, flat_stream, context_window=10)

app = PhraseTypingApp(registry, suggester, candidate_phrases, phrase_freq)
app.display()

## 3. The same workflow without the widget

Everything the widget does is also directly callable, which is useful for scripting or for following along here without clicking through the UI. We seed two types from domain knowledge (a "resumption" type at day scale, and a "decision formula" type at resolution scale), then ask for suggestions among the phrases neither type has claimed yet.

In [13]:
scripted_registry = PhraseTypeRegistry()

for p in ['luna den', 'martis den', 'mercurii den', 'jovis den', 'veneris den',
          'genoomen zyn geleesen en geresumeert', 'gelijk ook geresumeert en gearresteert']:
    scripted_registry.assign(p, 'resumption')
for p in ['waar op gedelibereert zynde is goedgevonden', 'is goedgevonden en verstaan dat',
          'waar op geen resolutie is gevallen']:
    scripted_registry.assign(p, 'decision formula')

print(scripted_registry.summary())

decision formula (3): ['is goedgevonden en verstaan dat', 'waar op gedelibereert zynde is goedgevonden', 'waar op geen resolutie is gevallen']
resumption (7): ['gelijk ook geresumeert en gearresteert', 'genoomen zyn geleesen en geresumeert', 'jovis den', 'luna den', 'martis den', 'mercurii den', 'veneris den']


In [14]:
uncategorised = scripted_registry.uncategorised(candidate_phrases)
print('uncategorised:', uncategorised)
print()

# Window choice matters and is deliberately not automatic here (see
# Document-Pattern-Detection-Findings.md): use a same-decision scale for
# the decision-formula type, a same-day scale for resumption.
for type_name, window in [('decision formula', 30), ('resumption', 300)]:
    members = scripted_registry.types[type_name].members
    suggestions = suggester.suggest_for_type(members, uncategorised, window=window, top_n=10)
    print(f'suggestions for {type_name!r} (window={window}):')
    for phrase, affinity, similarity in suggestions:
        print(f'  affinity={affinity:.3f}  context_sim={similarity:.3f}  {phrase}')
    print()

uncategorised: ['sabbathi den', 'ontfangen een missive van', 'is ter vergaderinge geleesen de requeste van']

suggestions for 'decision formula' (window=30):
  affinity=0.635  context_sim=0.708  ontfangen een missive van
  affinity=0.142  context_sim=0.792  is ter vergaderinge geleesen de requeste van
  affinity=0.000  context_sim=0.375  sabbathi den

suggestions for 'resumption' (window=300):
  affinity=0.773  context_sim=0.458  ontfangen een missive van
  affinity=0.335  context_sim=0.568  is ter vergaderinge geleesen de requeste van
  affinity=0.000  context_sim=0.337  sabbathi den



Both `ontfangen een missive van` and `is ter vergaderinge geleesen de requeste van` score moderately for *both* existing types but strongly for neither &mdash; which is the correct outcome, since they actually belong to a third, not-yet-created "opening formula" type. `sabbathi den` (the deliberately held-out sixth weekday name) scores low here too: it occurs only once in this corpus, so there isn't enough evidence either way &mdash; a useful reminder that very low-frequency candidates are inherently hard to place with any method, not a flaw specific to this one.

## 4. Browsing concordances directly

`concordance()` is the same function the widget's "Browse" tab and suggestion previews use under the hood.

In [15]:
for line in concordance('is ter vergaderinge geleesen de requeste van', flat_stream,
                        phrase_positions=cooc_index.phrase_positions, window=8, max_examples=5):
    print(' ', line)

  haar hoog mogende is deselve gehouden voor geleesen [is ter vergaderinge geleesen de requeste van] pieter kousemaker burger en inwoonder der stad aardenburg
  in conformiteit van de ordres van het land [is ter vergaderinge geleesen de requeste van] paulus vander vink rentmeester generaal van de abdye
  reekenkamer om te strekken tot der selver narigtinge [is ter vergaderinge geleesen de requeste van] n delforterie prelaat van de abdye van bandeloo
  van alles alhier ter vergaderinge rapport te doen [is ter vergaderinge geleesen de requeste van] mr hendrik hop haar hoog mog benoemde minister
  om in den haaren breeder gecommuniceert te worden [is ter vergaderinge geleesen de requeste van] verscheide kooplieden handelende in spaansche en portugaalsche wollen


## Notes

* `suggest_for_type`'s `window` parameter is deliberately manual, not auto-estimated. Every automatic window estimator built so far in this package becomes unreliable once a type's members are individually very frequent (no clean burst-vs-gap structure to exploit) &mdash; see `Document-Pattern-Detection-Findings.md`. Pick a window matching the scale you intend to capture for *that specific type*, and treat very different types (day-scale vs. decision-scale) as needing very different windows.
* `TypeSuggester.context_vectors()` plus `formula_detection.clustering.cluster_phrases` can cluster only the still-uncategorised phrases, to bootstrap discovery of types you haven't noticed yet, rather than only refining types you already have. This works better with a richer pool of uncategorised candidates than the handful used here.
* Because typing is multi-label, there's no tension between this workbench and the scale-based grouping in `Phrase-Recurrence-Scale-Grouping.ipynb`: a phrase's recurrence scale is one more piece of evidence you can look at while deciding which type(s) to assign it to, not a competing classification.